# Training Model CNN Custom untuk Aplikasi

Notebook ini melatih **satu model CNN Custom** (metode terbaik dari hasil eksperimen tugas) lalu menyimpannya untuk dipakai aplikasi.

Hasil yang disimpan ke Google Drive (`MyDrive/hasil_model.zip`):
- `model_cnn.keras` : model yang sudah dilatih
- `label_kelas.json` : daftar 50 kelas (kode folder mahasiswa)
- `contoh_uji/` : 1 citra uji per mahasiswa untuk demo aplikasi (tidak dipakai saat training)

Runtime: **GPU T4**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/MLA
!unzip -q "/content/drive/MyDrive/MLA.zip" -d /content/

## 1. Memuat dataset dan preprocessing

In [ ]:
import os, json, shutil, hashlib, numpy as np
from PIL import Image
from sklearn.preprocessing import LabelEncoder

DATA_DIR = "/content/MLA"
H, W = 64, 512
EXT = ('.jpg', '.jpeg', '.png')

paths, labels, seen = [], [], set()
for folder in sorted(os.listdir(DATA_DIR)):
    fp = os.path.join(DATA_DIR, folder)
    if not os.path.isdir(fp): continue
    for f in sorted(os.listdir(fp)):
        if not f.lower().endswith(EXT): continue
        p = os.path.join(fp, f)
        h = hashlib.md5(open(p, 'rb').read()).hexdigest()
        if h in seen: continue
        seen.add(h); paths.append(p); labels.append(folder)

def load_gray(p):
    img = Image.open(p).convert('L').resize((W, H), Image.BILINEAR)
    return np.array(img, dtype=np.uint8)

images = np.stack([load_gray(p) for p in paths])
le = LabelEncoder(); y = le.fit_transform(labels)

print("Jumlah total gambar :", len(images))
print("Jumlah kelas :", len(le.classes_))

## 2. Pembagian data

80% latih dan 20% uji (seed 0). Dari data latih, 15% disisihkan sebagai data validasi.

In [ ]:
from sklearn.model_selection import train_test_split

idx = np.arange(len(y))
idx_latih, idx_uji = train_test_split(idx, test_size=0.2, stratify=y, random_state=0)
idx_train, idx_val = train_test_split(idx_latih, test_size=0.15, stratify=y[idx_latih], random_state=0)

X_train, y_train = images[idx_train][..., None], y[idx_train]
X_val,   y_val   = images[idx_val][..., None],   y[idx_val]
X_uji,   y_uji   = images[idx_uji][..., None],   y[idx_uji]

print("Data latih :", len(X_train))
print("Data validasi :", len(X_val))
print("Data uji :", len(X_uji))

## 3. Training CNN Custom 

4 blok konvolusi (32 → 64 → 128 → 128 filter), learning rate 0,001, dropout 0,4, batch 32, maksimal 40 epoch dengan EarlyStopping.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
tf.keras.utils.set_random_seed(0)
print("GPU :", tf.config.list_physical_devices('GPU'))

NUM_CLASSES = len(le.classes_)

model = models.Sequential([
    layers.Input((H, W, 1)),
    layers.Rescaling(1/255.),
    layers.Conv2D(32, 3, padding='same'), layers.BatchNormalization(), layers.Activation('relu'), layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding='same'), layers.BatchNormalization(), layers.Activation('relu'), layers.MaxPooling2D(),
    layers.Conv2D(128, 3, padding='same'), layers.BatchNormalization(), layers.Activation('relu'), layers.MaxPooling2D(),
    layers.Conv2D(128, 3, padding='same'), layers.BatchNormalization(), layers.Activation('relu'),
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4),
    layers.Dense(NUM_CLASSES, activation='softmax')])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])

callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
             tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)]

hist = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                 epochs=40, batch_size=32, callbacks=callbacks, verbose=2)

## 4. Evaluasi pada data uji

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

p = model.predict(X_uji, verbose=0).argmax(1)
print("Akurasi   :", round(accuracy_score(y_uji, p), 4))
print("Precision :", round(precision_score(y_uji, p, average='macro', zero_division=0), 4))
print("Recall    :", round(recall_score(y_uji, p, average='macro', zero_division=0), 4))
print("F1-Score  :", round(f1_score(y_uji, p, average='macro', zero_division=0), 4))

## 5. Simpan model, label, dan contoh citra uji ke Google Drive

In [ ]:
OUT = "/content/hasil_model"
shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(os.path.join(OUT, "contoh_uji"))

model.save(os.path.join(OUT, "model_cnn.keras"))
with open(os.path.join(OUT, "label_kelas.json"), "w") as f:
    json.dump([str(k) for k in le.classes_], f)

# 1 citra uji per mahasiswa, nama file diawali kode kelas agar aplikasi bisa cek benar/salah
for kelas in le.classes_:
    i = next(i for i in idx_uji if labels[i] == kelas)
    ext = os.path.splitext(paths[i])[1].lower()
    shutil.copy(paths[i], os.path.join(OUT, "contoh_uji", f"{kelas}_contoh{ext}"))

shutil.make_archive("/content/hasil_model", "zip", OUT)
shutil.copy("/content/hasil_model.zip", "/content/drive/MyDrive/hasil_model.zip")

print("Tersimpan di Google Drive: MyDrive/hasil_model.zip")
print("Versi TensorFlow :", tf.__version__)